# Testing Efficiency of SQL vs. CSV tables

In this notebook, I will be performing several queries and assessing memory usage + time to achieve query using either the SQL tables or the csv tables in python

In [1]:
import assessment
from ExonPTMapper import config

## Memory

In [2]:

#
db_size = assessment.get_file_size('mapper.db')
print(f"Size of mapper.db: {db_size} MB")

mapper_folder_size = assessment.get_folder_size(config.processed_data_dir, file_type = '*.csv', files_to_ignore = ['alternative_ptms.csv', 'splice_events.csv'])
print(f"Total size of processed_data folder: {mapper_folder_size} MB")

#grab memory decrease
print(f"Memory decrease from CSV to SQL: {mapper_folder_size - db_size} MB ({((mapper_folder_size - db_size)/mapper_folder_size)*100:.2f}%)")



Size of mapper.db: 413.56640625 MB
Total size of processed_data folder: 1320.7561626434326 MB
Memory decrease from CSV to SQL: 907.1897563934326 MB (68.69%)


## Query Tests

In [3]:
import sqlite3
conn = sqlite3.connect('mapper.db')

from ExonPTMapper import mapping
mapper = mapping.PTM_mapper()

Loading mapper object from .tsv files
Loading exon-specific data
Loading transcript-specific data
Loading gene-specific info
Loading unique protein isoforms
Loading protein-specific info
Loading information on PTMs on canonical proteins
Loading genomic coordinates of PTMs associated with canonical proteins
Loading information on PTMs projected onto variant proteins
Loading information on PTMs on unique protein isoforms


###  Convert gene name to SwissProt ID

In [ ]:


def DB_get_swissprot_id(gene_name):
    query = """
    SELECT gene_to_proteins.SwissProt_ID
    FROM genes
    JOIN gene_to_proteins ON genes.Gene_stable_ID = gene_to_proteins.Gene_stable_ID
    WHERE genes.Gene_name = ?
    """
    cursor = conn.execute(query, (gene_name,))
    swissprot_id = cursor.fetchone()[0]
    return swissprot_id

def PY_get_swissprot_id(gene_name):
    prot_id = config.translator[config.translator['Gene name'] == gene_name].dropna(subset = 'UniProtKB/Swiss-Prot ID')['UniProtKB/Swiss-Prot ID'].values[0]
    return prot_id

In [ ]:
db_query_time = assessment.time_function(DB_get_swissprot_id, num_calls = 1000, gene_name = 'EGFR')

Current memory usage is 0.017166MB; Peak was 0.018178MB
Elapsed time for 1000 calls: 1.2923648357391357 seconds


In [ ]:
py_query_time = assessment.time_function(PY_get_swissprot_id, num_calls = 1000, gene_name = 'EGFR')
print(f"Python query time for gene name to SwissProt ID (1000 calls): {py_query_time} seconds")

Current memory usage is 0.364662MB; Peak was 0.961462MB
Elapsed time for 1000 calls: 19.416130542755127 seconds
Python query time for gene name to SwissProt ID (1000 calls): 19.416130542755127 seconds


### Get non-constitutive PTMs for a given protein

In [16]:
import pandas as pd
def DB_get_ptm_table(swissprot_id, include_constitutive = False):
    query = f"""SELECT Residue, Position, Modification_Class, ptm_conservation_score 
                FROM known_ptms WHERE SwissProt_ID = ?"""
    if not include_constitutive:
        query += " AND ptm_conservation_score < 1"

    query += " ORDER BY Position"
    results = conn.execute(query, (swissprot_id,)).fetchall()


    #construct and output table
    results = pd.DataFrame(results, columns=["Residue", "Position", "Modification_Class", "ptm_conservation_score"])
    return results


mapper.ptm_info['Swiss-Prot ID'] = mapper.ptm_info['Protein'].str.split('-').str[0]
def PY_get_ptm_table(mapper, swissprot_id, include_constitutive = False):


    ptms = mapper.ptm_info[mapper.ptm_info['Swiss-Prot ID'] == swissprot_id]
    if not include_constitutive:
        ptms = ptms[ptms['PTM Conservation Score'] < 1]
    ptms = ptms.sort_values(by = 'PTM Location (AA)')

    ptms = ptms[['Residue', 'PTM Location (AA)', 'Modification Class', 'PTM Conservation Score']]
    return ptms

In [20]:
print('Database method:')
elapsed_time = assessment.time_function(DB_get_ptm_table, num_calls = 1000, swissprot_id = 'P00533')
print('\nPython/CSV method:')
elapsed_time = assessment.time_function(PY_get_ptm_table, num_calls = 1000, mapper = mapper, swissprot_id = 'P00533')

Database method:
Current memory usage is 0.190537MB; Peak was 0.224887MB
Elapsed time for 1000 calls: 1.1071033477783203 seconds

Python/CSV method:
Current memory usage is 0.161963MB; Peak was 0.605348MB
Elapsed time for 1000 calls: 23.800536394119263 seconds
